# Post-Training与遗忘总结

## 概述

在当前的通用模型时代，post-training（后训练）成为一种常见情景，它指的是基于预训练的基础模型进行进一步的特定任务优化。这种做法能够显著提升目标任务的表现，但也可能带来对其他任务性能下降的风险，即所谓的“灾难性遗忘”。


## ​一、核心问题：灾难性遗忘（Catastrophic Forgetting）​
- 定义：模型在持续学习新任务时，意外丢失已掌握的旧任务能力，类比“手术成功但病人死亡”。

- 根本原因​​：
    - ​参数覆盖​​：新任务训练覆盖旧任务的关键权重。
    - ​任务冲突​​：新旧任务差异越大（如文本→语音），遗忘越严重。
    - ​数据偏差​​：微调数据分布偏离预训练数据，导致模型行为偏移。
- 典型场景​​：
    - ​语言能力丢失​​（如教LLaMA-2中文后英文能力退化）。
    - ​安全对齐失效​​（微调后模型输出有害内容，如银行密码攻击建议）。
    - ​多模态冲突​​（添加语音理解能力后，文本处理能力受损）

## 二、实证影响与关键案例​

### （1）基础能力退化
- ​​数学与推理​​：
    - 微调代码生成模型（Magicoder-Evol）后，HumanEval得分下降6.7%。
    - 数学任务（GSM8K）微调导致通用推理能力平均降低5.3%。
- ​​安全性能崩塌​​：
    - LLaMA-2-Chat微调后，ToxiGen毒性分数从0.22升至5.74（26倍增长）。
    - 对齐模型（如Llama-3-8B）微调后绕过安全约束的概率提升37%。
### （2）多模态任务冲突
- ​​语音-文本冲突​​：
    - 训练语音理解任务（如情感识别）后，模型丧失基础JSON格式输出能力。
    - Dynamic SUPERB基准测试显示：添加23个语音任务，文本任务准确率平均下降12.4%。
- ​​跨模态干扰机制​​：
    - 语音编码器（Whisper）与文本解码器的参数竞争导致特征空间扭曲。

## 三、缓解技术与发展脉络​
### 经验重放（Experience Replay）
- ​​核心原理​​：训练新任务时混入5%旧任务数据。
- ​​有效性​​：
    - LAMOL框架验证：多任务性能差距从16.7%缩小至2.3%。
    - 安全微调中，毒性分数回升率达89%。
- ​​局限​​：依赖旧任务数据（常不可得，如RLHF对齐数据）。

### 自合成重放（Self-Synthesized Rehearsal, SSR）
- ​​突破性方案​​：用当前模型生成旧任务的伪样本。
    - ​​重述优化​​：将问题改写为模型友好格式（如简化语言）。
- ​​关键成果​​：
    - SDFT方法在OpenFunctions任务上提升5.0%准确率。
    - 伪数据训练使遗忘率降低至2.0%（vs. 标准微调12.7%）。

### 参数高效微调（PEFT）
- ​​LoRA​​：
    - 冻结原权重，仅更新低秩适配器。
    - TruthfulQA毒性分数仅0.79（vs. 全参数微调5.74）。
- ​​(IA)³技术​​：
    - 注入任务特定参数，安全微调后毒性分数保持0.17。

## ​四、前沿研究与未来方向​​
### ​​动态评估体系​​：
- Dynamic SUPERB Phase-2扩展至180个语音-文本任务，量化多模态遗忘。
- AIR-Bench-Chat评估框架整合安全、推理、多模态三维指标。

## ​​跨模型协作架构​​：
- ​​DeSTA2框架​​：分离语音编码器（Whisper）与文本解码器，冲突降低40%。
- ​​Qwen2-Audio​​：通过模态隔离设计，综合得分达51.69（当前最优）。

## ​训练策略革命​​：
- ​语言适配优化​​：生成“模型友好”伪数据（如缩短输出长度），遗忘率再降34%。
- ​RLHF融合方案​​：初步证据显示强化学习微调遗忘较少（机制待探索）。

## 五、实践指导原则​

### 1、​技术选型优先级​​：
- SSR自合成重放 > LoRA > 全参数微调  （当旧数据缺失时，SSR为最优解）
​
### 2、​部署前验证​​：
- 必须测试TruthfulQA、Dynamic SUPERB等多任务基准。
- 安全关键场景需额外进行对抗性Prompt测试。
​​
### 3、​开源资源推荐​​：
- LAMOL代码库（持续学习框架）。
- BLSP工具链（语音-文本多模态微调）。

## ​​核心结论​​：

- 灾难性遗忘是后训练的核心挑战，但通过自合成数据、参数隔离和动态评估，可构建“既学新知识，又保旧能力”的稳健模型。未来需在跨模态架构和安全对齐机制上持续突破。
- Post-training 大模型容易遗忘过去的技能，用大模型自己说的话来做Post-training